# Prototipo — Asignación de vueltas con ventanas y tráfico (Grupo 4)
Cuaderno de demostración para la Entrega 1. En Google Colab: suba el repositorio (o clónelo) y ejecute las celdas en orden.

In [ ]:
# En Colab: !git clone <URL-del-repositorio> && %cd transporte-cusco-scheduling
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, 'src')

In [ ]:
from sched.traffic import get_profile
for n in ('T0','T1','T2'):
    p = get_profile(n)
    print(n, [round(b.factor,3) for b in p.bands], 'media =', round(p.mean_factor(),3))
print('J1 sale 06:00, base 90 min, T2 ->', round(get_profile('T2').travel_time(360, 90), 2), 'min')

In [ ]:
!python -m sched build
!python -m sched demo --traffic T2

In [ ]:
from sched.model import Instance
from sched.algorithms import list_scheduling, lpt
from sched.validator import validate
from sched.metrics import compute_metrics
from sched.viz import gantt
from IPython.display import Image
inst = Instance.load('data/instances/RTI-01.json'); t1 = get_profile('T1')
for alg in (list_scheduling, lpt):
    s = alg(inst, t1); m = compute_metrics(s, t1)
    print(s.algorithm, 'Lmax=%.2f h' % m['Lmax_h'], 'no asignadas=', m['no_asignados'], 'violaciones=', len(validate(s, t1)))
gantt(list_scheduling(inst, t1), t1, 'results/gantt_notebook.png')
Image('results/gantt_notebook.png')

In [ ]:
from sched.simulation import simulate
s = list_scheduling(inst, t1)
for sigma in (0.1, 0.2):
    r = simulate(s, t1, sigma, reps=200)
    print(f"σ={sigma}: {r['vueltas_fuera_ventana_pct']:.1f} % fuera de ventana, retraso medio {r['retraso_medio_min']:.1f} min")

In [ ]:
!python -m pytest -q